# 07 · Evaluation on the held-out women

Everything here is measured on the 1,375 held-out women, with their survey weights, and every
interval comes from resampling **whole clusters within their strata** (500 times), so it
reflects the survey's design.

For each model and rung: average precision against chance (the share reaching the rung),
AUROC, the Brier score, calibration (slope and intercept), the share of the women one is
looking for who sit in the top fifth of the ranking, sensitivity and specificity at the
threshold chosen on the development women, AUROC within regions, wealth, education and
residence groups, the difference from the best model and from logistic regression with its
paired interval, and the net benefit across risk thresholds (decision curve).

Two further checks: every model is refitted from its saved setting and must reproduce its
saved predictions exactly, and the tuning is examined: what the grid gained over the model's
default settings, and how much each setting matters.

In [ ]:
# Settings for this notebook
QUICK = False   # True: fewer resamples for the intervals (a check that everything runs)

import sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
here = Path.cwd()
sys.path.insert(0, str(here.parent / "src" if here.name.startswith("notebooks") else here / "src"))
from anc8 import repro
P = repro.setup(quick=QUICK)
QUICK = P.quick

import numpy as np
import pandas as pd
from anc8 import library as lib

MODELS = ['Logistic regression', 'Random Forest', 'XGBoost']

In [ ]:
repro.run_script("fit_library_models.py", "--root", P.results)
repro.run_script("run_hyperparameter_ablation.py", "--root", P.results)
repro.run_script("run_explainability.py")   # the cards show what each model leans on (notebook 08)
repro.run_script("run_model_cards.py", "--library", P.results, *(["--quick"] if QUICK else []))

## The three models at the three rungs

In [ ]:
cards = json.load(open(P.outputs / "model_cards" / "model_cards.json"))
rows = []
for key, by in cards["cards"].items():
    fam = cards["families"][key]
    for rung, c in by.items():
        rows.append({"rung": rung, "model": fam["name"], "AP": c["ap"], "AP 95%": f"{c['ap_ci'][0]:.3f}-{c['ap_ci'][1]:.3f}",
                     "chance": c["prevalence"], "AUROC": c["auroc"], "slope": c["slope"], "top 20% hold": c["top20"]})
cardtab = pd.DataFrame(rows).sort_values(["rung", "AP"], ascending=[True, False])
print(cardtab.round(3).to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
for ax, rung in zip(axes, ["anc1", "anc4", "anc8"]):
    t = cardtab[cardtab.rung == rung].set_index("model")
    ax.bar(t.index, t.AP, color=["#4c78a8", "#72b7b2", "#f58518"])
    ax.axhline(t.chance.iloc[0], color="grey", lw=0.8, ls="--")
    ax.set_title(rung.upper() + "+"); ax.set_ylim(0, 1); ax.tick_params(axis="x", labelrotation=20)
axes[0].set_ylabel("held-out average precision (dashed: chance)")
plt.tight_layout(); plt.show()

## What tuning bought

In [ ]:
for rung in lib.RUNGS:
    a = pd.read_csv(P.outputs / "library" / "tuning" / f"ablation_s21_{rung}.csv")
    g = a[a.parameter.isin(["(library defaults)", "(tuned optimum)"])].pivot_table(index="model", columns="parameter", values="AP")
    g["gain"] = g["(tuned optimum)"] - g["(library defaults)"]
    print(rung); print(g.round(4).to_string(), "\n")
repro.compare_cells(P.results, MODELS)
repro.report()